# SDXL Base 1.0 · GenEval 2

In [ ]:
import hashlib
import json
import os
import zipfile
from pathlib import Path

project = "sdxl"
archive_name = "replaceme-sdxl-env.zip"
expected_manifest = "45e7d06d2131ae2e6995989d5c8770aed4027f8a70882294e5b47259f0f1b01c"
kaggle = Path("/kaggle/input").is_dir()
default_input = Path("/kaggle/input/replaceme-sdxl-env") if kaggle else Path.cwd()
inputs = Path(os.environ.get("REPLACEME_INPUT", default_input)).expanduser().resolve()
work = Path(os.environ.get("REPLACEME_WORK", "/kaggle/working" if kaggle else "replaceme-work")).resolve()

archive_path = inputs if inputs.name == archive_name else inputs / archive_name
if archive_path.is_file():
    bundle = archive_path
    with zipfile.ZipFile(bundle) as archive:
        metadata = archive.read("environment.json")
        bootstrap = archive.read("bootstrap.py")
elif inputs.is_dir() and (inputs / "environment.json").is_file():
    bundle = inputs
    metadata = (bundle / "environment.json").read_bytes()
    bootstrap = (bundle / "bootstrap.py").read_bytes()
else:
    raise FileNotFoundError(f"Ожидается {archive_path}. Укажите точный путь к ZIP или каталогу Dataset в REPLACEME_INPUT.")

if hashlib.sha256(metadata).hexdigest() != expected_manifest:
    raise ValueError(f"Манифест не соответствует комплекту {archive_name}.")
manifest = json.loads(metadata)
if manifest["project"] != project or hashlib.sha256(bootstrap).hexdigest() != manifest["bootstrap_sha256"]:
    raise ValueError("Ноутбук и загрузчик окружения относятся к разным сборкам.")
scope = {}
exec(compile(bootstrap, "bootstrap.py", "exec"), scope)
scope["start"](bundle, manifest, work / project)


## Промпты и параметры

In [ ]:
%%benchmark
import gc, json, os, sys
from pathlib import Path
from statistics import mean
from tempfile import mkdtemp

import torch
from PIL import Image
from diffusers import StableDiffusionXLPipeline
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download, snapshot_download
from scipy.stats import gmean
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

if not torch.cuda.is_available():
    raise RuntimeError("Включите GPU в Settings → Accelerator.")
torch.set_num_threads(2)
data_id, data_rev = "Jialuo21/GenEval2", "f54cd910012a5476120ed82706fac964689eda9f"
data_path = hf_hub_download(data_id, "data/test-00000-of-00001.parquet", repo_type="dataset", revision=data_rev)
dataset = pq.read_table(data_path).to_pylist()
indices = [2, 46, 126, 233, 345, 694]
items = {i: dataset[i] for i in indices}
run = Path(mkdtemp(prefix="sdxl-geneval2-", dir=Path.cwd()))
sdxl_id, sdxl_rev = "stabilityai/stable-diffusion-xl-base-1.0", "462165984030d82259a11f4367a4eed129e94a7b"
judge_id, judge_rev = "Qwen/Qwen3-VL-8B-Instruct", "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b"
seed, steps, cfg, size = 0, 50, 5.0, 1024

def weights(repo, revision, patterns, override):
    if folder := os.getenv(override):
        return folder
    return snapshot_download(repo, revision=revision, allow_patterns=patterns)

print(torch.cuda.get_device_name(), "· промптов", len(items), "· вопросов", sum(len(x["vqa_list"]) for x in items.values()))

## Генерация SDXL

In [ ]:
%%benchmark
patterns = ["model_index.json", "scheduler/*.json", "tokenizer*/*", "text_encoder*/*.json",
            "text_encoder*/model.fp16.safetensors", "unet/config.json", "unet/*.fp16.safetensors",
            "vae/config.json", "vae/*.fp16.safetensors"]
model_path = weights(sdxl_id, sdxl_rev, patterns, "SDXL_MODEL")
pipe = StableDiffusionXLPipeline.from_pretrained(model_path, variant="fp16", torch_dtype=torch.float16,
                                                use_safetensors=True, vae=None, add_watermarker=False, local_files_only=True)
pipe.set_progress_bar_config(disable=True)
pipe.text_encoder.to("cuda")
pipe.text_encoder_2.to("cuda")
with torch.inference_mode():
    embeddings = {i: pipe.encode_prompt(item["prompt"], device=torch.device("cuda")) for i, item in items.items()}
pipe.register_modules(text_encoder=None, text_encoder_2=None)
gc.collect()
torch.cuda.empty_cache()

pipe.to("cuda")
scheduler = type(pipe.scheduler).__name__
with torch.inference_mode():
    latents = {}
    for i, (pos, neg, pool, neg_pool) in embeddings.items():
        latents[i] = pipe(prompt_embeds=pos, negative_prompt_embeds=neg, pooled_prompt_embeds=pool,
                          negative_pooled_prompt_embeds=neg_pool, height=size, width=size,
                          num_inference_steps=steps, guidance_scale=cfg, output_type="latent",
                          generator=torch.Generator(device="cpu").manual_seed(seed)).images.cpu()
        print(f"{i:03d}  {items[i]['prompt']}", flush=True)
del pipe, embeddings
gc.collect()
torch.cuda.empty_cache()

In [ ]:
%%benchmark
from diffusers import AutoencoderKL
from diffusers.image_processor import VaeImageProcessor

vae = AutoencoderKL.from_pretrained(model_path, subfolder="vae", variant="fp16", torch_dtype=torch.float32,
                                    local_files_only=True).to("cuda")
processor = VaeImageProcessor(vae_scale_factor=8)
with torch.inference_mode():
    for i, z in latents.items():
        decoded = vae.decode(z.to("cuda", dtype=vae.dtype) / vae.config.scaling_factor).sample
        processor.postprocess(decoded, output_type="pil")[0].save(run / f"{i:04d}.png")
del vae, decoded, latents
gc.collect()
torch.cuda.empty_cache()

## GenEval 2 · Soft-TIFA

In [ ]:
%%benchmark
from tempfile import TemporaryDirectory
import psutil

judge_path = weights(judge_id, judge_rev, ["*.json", "*.safetensors", "*.txt", "*.jinja"], "GENEVAL_JUDGE")
processor = AutoProcessor.from_pretrained(judge_path, local_files_only=True)
judge_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
gib = 1024 ** 3
memory = {0: max(gib, torch.cuda.mem_get_info()[0] - 2 * gib),
          "cpu": max(gib, int(psutil.virtual_memory().available * .50))}
offload = TemporaryDirectory(prefix="sdxl-judge-", dir=Path.cwd())
judge = Qwen3VLForConditionalGeneration.from_pretrained(
    judge_path, dtype=judge_dtype, device_map="auto", max_memory=memory,
    offload_folder=offload.name, offload_buffers=True, local_files_only=True).eval()

@torch.inference_mode()
def soft_tifa(image_path, question, answer):
    allowed = ["Yes", "yes", " yes", " Yes"]
    if question.startswith("How many"):
        words = "one two three four five six seven eight nine ten".split()
        digit = str(words.index(answer) + 1) if answer in words else "other"
        allowed = [answer, answer.capitalize(), " " + answer, " " + answer.capitalize(), digit, " " + digit]
    message = [{"role": "user", "content": [{"type": "image", "image": str(image_path)},
               {"type": "text", "text": f"{question} Answer in one word."}]}]
    inputs = processor.apply_chat_template(message, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(judge.device)
    result = judge.generate(**inputs, max_new_tokens=1, do_sample=False,
                            output_scores=True, return_dict_in_generate=True)
    probabilities = result.scores[0][0].softmax(dim=-1)
    return sum(probabilities[processor.tokenizer.encode(a)[0]].item() for a in allowed)

In [ ]:
%%benchmark
rows = []
for i, item in items.items():
    path = run / f"{i:04d}.png"
    scores = [{"question": q, "answer": a, "skill": skill, "score": soft_tifa(path, q, a)}
              for (q, a), skill in zip(item["vqa_list"], item["skills"], strict=True)]
    values = [s["score"] for s in scores]
    rows.append(dict(index=i, prompt=item["prompt"], image=path.name, questions=scores,
                     am_pct=100 * mean(values), gm_pct=100 * float(gmean(values))))
    print(f"{i:03d}  AM {rows[-1]['am_pct']:6.2f}%   GM {rows[-1]['gm_pct']:6.2f}%", flush=True)
del judge
gc.collect()
torch.cuda.empty_cache()
offload.cleanup()

## Результат

In [ ]:
%%benchmark
from importlib.metadata import version
from IPython.display import Markdown, display

summary = dict(model=sdxl_id, model_revision=sdxl_rev, judge=judge_id, judge_revision=judge_rev,
               dataset_revision="a6e82d2289e8d418f27f0adee77908b07060eea3", dataset_size=len(dataset),
               dataset_hub=data_id, dataset_hub_revision=data_rev,
               indices=list(items), questions=sum(len(r["questions"]) for r in rows),
               seed=seed, steps=steps, guidance_scale=cfg, resolution=size, scheduler=scheduler,
               gpu=torch.cuda.get_device_name(), judge_dtype=str(judge_dtype),
               versions={p: version(p) for p in ("torch", "diffusers", "transformers", "accelerate")},
               soft_tifa_am_pct=mean(r["am_pct"] for r in rows), soft_tifa_gm_pct=mean(r["gm_pct"] for r in rows))
for name, data in [("summary", summary), ("scores", rows)]:
    (run / f"{name}.json").write_text(json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False))
display(Markdown(f"**{len(rows)} изображений · AM {summary['soft_tifa_am_pct']:.2f}% · GM {summary['soft_tifa_gm_pct']:.2f}%**"))
from PIL import ImageDraw
gallery = Image.new("RGB", (720, 264 * ((len(rows) + 2) // 3)), "white")
for n, row in enumerate(rows):
    x, y = (n % 3) * 240, (n // 3) * 264
    with Image.open(run / row["image"]) as image:
        gallery.paste(image.resize((240, 240), Image.Resampling.LANCZOS), (x, y))
    ImageDraw.Draw(gallery).text((x + 4, y + 244), f"#{row['index']}   AM {row['am_pct']:.1f}   GM {row['gm_pct']:.1f}", fill="black")
gallery.save(run / "gallery.jpg", quality=90)
from IPython.display import Image as Preview
display(Preview(filename=str(run / "gallery.jpg")))
print("PNG и JSON ·", run)